In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git clone https://github.com/abachaa/MedQuAD.git //content/drive/MyDrive/Medical_Q&A_Chatbot

/bin/bash: line 1: A_Chatbot: command not found
Cloning into '//content/drive/MyDrive/Medical_Q'...
remote: Enumerating objects: 11310, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 11310 (delta 7), reused 5 (delta 5), pack-reused 11300 (from 1)
Receiving objects: 100% (11310/11310), 11.01 MiB | 8.69 MiB/s, done.
Resolving deltas: 100% (6807/6807), done.
Updating files: 100% (11277/11277), done.


In [ ]:
!pip install streamlit pandas sentence-transformers faiss-cpu spacy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 109.0 MB/s eta 0:00:00


In [ ]:
!pip install scispacy spacy -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 32.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but 

In [ ]:
pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 97.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.0/865.0 kB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 101.2 MB/s eta 0:00:00
  Created wheel for en_core_sci_sm: filename=en_core_sci_sm-0.5.4-py3-none-any.whl size=14778488 sha256=384f50b50e48fba804de7825bb8fb9fdc06384ba5e2b776e2fb3f72c7030dae6
  Stored in directory: /root/.cache/pip/wheels/49/7f/0f/ec0fc3a935bfe55e6ef2ca04b7a31e33cbd533a6d7cbd9e11e
Successfully built en_core_sci_sm
  Attempting uninstall: blis
    Found existing installation: blis 1.3.3
    Uninstalling blis-1.3.3:
      Successfully uninstalled blis-1.3.3
  Attempting uninstall: con

In [ ]:
%%writefile app.py


import os


import glob
import xml.etree.ElementTree as ET
import pandas as pd
import streamlit as st
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import spacy

# --- CONFIG & SETUP ---
st.set_page_config(page_title="Medical Q&A Chatbot", page_icon="🩺", layout="wide")


# Change this line in your app.py
MEDQUAD_PATH = "/content/MedQuAD_Local"

# --- 1. DATA LOADING & PARSING ---
@st.cache_data(show_spinner="Parsing MedQuAD dataset...")
def load_data(base_path, sample_size=5000):
    """Parses XML files from MedQuAD and returns a DataFrame."""
    data = []
    # Find all XML files in subdirectories

    xml_files = glob.glob(os.path.join(base_path, "**", "*.xml"), recursive=True)

    for xml_file in xml_files:
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
            # Iterate through QAPairs
            for qa in root.findall('.//QAPair'):
                question = qa.find('Question')
                answer = qa.find('Answer')
                if question is not None and answer is not None and question.text and answer.text:
                    data.append({
                        'question': question.text.strip(),
                        'answer': answer.text.strip()
                    })
        except Exception as e:
            continue

    df = pd.DataFrame(data)
    # We sample the data to save RAM/Time for this demonstration
    if len(df) > sample_size:
        df = df.sample(sample_size, random_state=42).reset_index(drop=True)
    return df

# --- 2. RETRIEVAL MECHANISM (VECTOR DB) ---
@st.cache_resource(show_spinner="Loading Embedding Model and Indexing Data...")
def setup_retrieval_system(df):
    """Embeds questions and creates a FAISS index for fast retrieval."""
    # Load a lightweight, fast sentence transformer
    embedder = SentenceTransformer('all-MiniLM-L6-v2')

    # Embed all questions in the dataset
    questions = df['question'].tolist()
    embeddings = embedder.encode(questions, convert_to_numpy=True)

    # Create FAISS Index
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return embedder, index

# --- 3. MEDICAL ENTITY RECOGNITION ---
@st.cache_resource(show_spinner="Loading Medical NER Model...")
def load_ner_model():
    """Loads scispacy model for medical entity recognition."""
    # en_core_sci_sm is specialized for biomedical text
    return spacy.load("en_core_sci_sm")

def extract_entities(text, nlp_model):
    doc = nlp_model(text)
    entities = [ent.text for ent in doc.ents]
    return list(set(entities))

# --- MAIN APP LOGIC ---
def main():
    st.title("🩺 Medical Q&A Chatbot")
    st.markdown("*Powered by the MedQuAD Dataset and Semantic Search*")
    st.warning("**Disclaimer:** This is an experimental system for educational purposes only. Do not use for actual medical diagnostics.")

    # Initialize components
    if not os.path.exists(MEDQUAD_PATH):
        st.error(f"MedQuAD dataset not found at {MEDQUAD_PATH}. Please clone the repository first.")
        return

    df = load_data(MEDQUAD_PATH)
    embedder, index = setup_retrieval_system(df)
    nlp = load_ner_model()

    # Session state for chat history
    if "messages" not in st.session_state:
        st.session_state.messages = []

    # Display chat history
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])
            if "entities" in msg and msg["entities"]:
                st.caption(f"🧬 Detected Medical Entities: {', '.join(msg['entities'])}")

    # Chat Input
    if user_query := st.chat_input("Ask a medical question (e.g., 'What are the symptoms of asthma?'):"):

        # 1. Add user message to state and UI
        st.session_state.messages.append({"role": "user", "content": user_query})
        with st.chat_message("user"):
            st.markdown(user_query)

        # 2. Extract Entities
        entities = extract_entities(user_query, nlp)

        # 3. Retrieve relevant answer
        query_embedding = embedder.encode([user_query], convert_to_numpy=True)
        distances, indices = index.search(query_embedding, k=1) # Get top 1 result

        best_match_idx = indices[0][0]
        distance = distances[0][0]

        # Fetch the corresponding answer and matched question
        matched_question = df.iloc[best_match_idx]['question']
        matched_answer = df.iloc[best_match_idx]['answer']

        # Format the response
        response = f"**Closest matched question:** _{matched_question}_\n\n**Answer:**\n{matched_answer}"

        # If the distance is too high, the model isn't confident
        if distance > 1.5: # Threshold can be tuned
            response = "I couldn't find a highly confident answer in my database. However, here is the closest information I found:\n\n" + response

        # 4. Add assistant response to state and UI
        st.session_state.messages.append({"role": "assistant", "content": response, "entities": entities})
        with st.chat_message("assistant"):
            if entities:
                st.caption(f"🧬 Detected Medical Entities: {', '.join(entities)}")
            st.markdown(response)

if __name__ == "__main__":
    main()

Overwriting app.py


In [ ]:
# This copies the folder from Drive to the local Colab disk
!cp -r "/content/drive/MyDrive/Medical_Q&A_Chatbot/MedQuAD" "/content/MedQuAD_Local"

In [ ]:
# 1. Install Cloudflared
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb



--2026-04-20 03:57:24--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb [following]
--2026-04-20 03:57:24--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ec689fe1-d727-4ebd-bbc3-5967730ab54e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-20T04%3A44%3A01Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [ ]:
# 2. Run your app (Note: adding flags to ignore CORS/XSRF helps tunneling)
import os
os.system("streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false &")

# 3. Create the stable tunnel
!cloudflared tunnel --url http://localhost:8501

2026-04-20T03:57:40Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-20T03:57:40Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-20T03:57:42Z INF +--------------------------------------------------------------------------------------------+
2026-04-20T03:57:42Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-20T03:57:42Z INF |  https://bundle-badly-basically-reading.trycloudflare.